<a href="https://colab.research.google.com/github/BarbaraEstimable/IA2_projet1/blob/Algorithmes-de-Dijkstra/projet1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projet 1 - Plus Court Chemin entre Villes




###Contexte

---



On souhaite modéliser un réseau routier entre plusieurs villes nord-américaines
sous la forme d’un graphe orienté pondéré. Chaque ville est un noeud, chaque
route entre deux villes est un arc orienté dont le poids représente la distance en
kilomètres.
L’objectif est de trouver le plus court chemin entre une ville de départ et
une ville d’arrivée, en utilisant des algorithmes de plus court chemin, puis de
comparer leurs performances.

###Objectif

---



À partir du jeu de données fourni ci-dessous (villes, coordonnées, routes et distances),
implémenter une structure de graphe orienté pondéré, puis utiliser
au moins deux algorithmes parmi Dijkstra, A* et Bellman-Ford pour trouver
le plus court chemin entre deux villes.
Pour chacun des algorithmes, afficher dans la console les performances (nombre
de noeuds explorés, coût total du chemin, temps d’exécution, …).

###Directives

---





*   Nous travaillerons avec un graphe orienté pondéré (le poids d’un arc
représente une distance en km entre deux villes)
*   Une Ville est un objet possédant un nom et des coordonnées géographiques
(x, y) utilisées pour le calcul de l’heuristique de A*


*   Un Arc est orienté : si une route Montréal → Toronto existe avec un
poids donné, cela ne signifie pas que Toronto → Montréal existe avec le
même poids
*   Le jeu de données est entièrement fourni : vous n’avez pas à inventer
les villes, coordonnées ou distances


*   Une implémentation graphique devra être réalisée : visualisation du
graphe avec les arcs fléchés, les poids, et le chemin solution mis en évidence.
(Utilisation de la librairie de votre choix, Graphviz ou Matplotlib ou autre)
*   Implémenter au moins 2 algorithmes parmi : Dijkstra, A*, Bellman-
Ford


*   Heuristique de A* : Distance Euclidienne entre les coordonnées (x, y)
des villes
*   Choisir au moins 3 métriques pertinentes : nombre de noeuds explorés,
coût total du chemin (km), temps d’exécution, …


*   Une analyse des résultats est attendue : discuter des différences de
performance entre les algorithmes, et expliquer dans quels cas chacun est
préférable











 **Pour l’analyse des résultats prenez le chemin entre
Québec et Buffalo**

### Données fournies

In [ ]:
VILLES = {
  "Montreal": (45.30, -73.35),
  "Quebec": (46.81, -71.21),
  "Ottawa": (45.42, -75.70),
  "Toronto": (43.65, -79.38),
  "Buffalo": (42.89, -78.87),
  "Boston": (42.36, -71.06),
  "New York": (40.71, -74.01),
  "Chicago": (41.88, -87.63),
}

ROUTES = [
  # Depuis Montreal
  ("Montreal", "Quebec", 250),
  ("Montreal", "Ottawa", 200),
  ("Montreal", "Boston", 435),
  ("Montreal", "New York", 595),

  # Depuis Quebec
  ("Quebec", "Montreal", 255),
  ("Quebec", "Boston", 650),

  # Depuis Ottawa
  ("Ottawa", "Montreal", 195),
  ("Ottawa", "Toronto", 450),

  # Depuis Toronto
  ("Toronto", "Ottawa", 445),
  ("Toronto", "Buffalo", 155),
  ("Toronto", "Chicago", 840),

  # Depuis Buffalo
  ("Buffalo", "Toronto", 160),
  ("Buffalo", "New York", 590),
  ("Buffalo", "Boston", 700),
  ("Buffalo", "Chicago", 860),

  # Depuis Boston
  ("Boston", "New York", 345),
  ("Boston", "Montreal", 440),
  ("Boston", "Buffalo", 695),

  # Depuis New York
  ("New York", "Boston", 350),
  ("New York", "Buffalo", 585),
  ("New York", "Chicago", 1270),

  # Depuis Chicago
  ("Chicago", "Toronto", 835),
  ("Chicago", "Buffalo", 855),
  ("Chicago", "New York", 1275),
]

### Importations

In [ ]:
import heapq
import time
import matplotlib.pyplot as plt
import math

### Les class

In [ ]:
class Ville:
    def __init__(self, nom, x, y):
        self.nom = nom
        self.x = x  # longitude approximative en degrés
        self.y = y  # latitude approximative en degrés


class Graphe:
    def __init__(self, liste_villes):
        graphe_liste_adjacence = {}
        for ville in liste_villes:
            graphe_liste_adjacence.update({ville : []})

        self.graphe_liste_adjacence = graphe_liste_adjacence
        self.villes = {}

    def ajoute_ville(self, nom, x, y):
        self.villes[nom] = Ville(nom, x, y)

    def ajoute_arc(self, noeud1, noeud2, poids):
        self.graphe_liste_adjacence[noeud1].append(noeud2, poids)


    def sont_voisins(self, noeud1, noeud2) -> bool:
        return any(v==noeud2 for v, _ in self.graphe_liste_adjacence[noeud1])

    def voisins(self, noeud) -> list:
        return self.graphe_liste_adjacence[noeud]


In [ ]:
# La construction du graphe
def construction_graphe():
    liste_villes = list(VILLES.keys())
    graphe = GrapheO(liste_villes)
    for origine, destination, distance in ROUTES:
        graphe.ajoute_ville(origine, destination, distance)
    return graphe

In [ ]:
# Heuristique de A*
   def heuristique(self, ville_depart, ville_arrivee):
        x1, y1 = self.villes[ville_depart].x, self.villes[ville_depart].y
        x2, y2 = self.villes[ville_arrivee].x, self.villes[ville_arrivee].y
        return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

In [ ]:
# Le chemin
def remonte_chemin(depart, arrivee, parent):
    sommet = arrivee
    chemin = [arrivee]
    while sommet != depart:
        sommet = parent[sommet]
        chemin.append(sommet)
    chemin.reverse()
    return chemin


### Algorithme Dijkstra

In [ ]:
def dijkstra(grille, depart, arrivee):
    # Le temps
    debut = time.perf_counter()

    # Initialisation des distances
    distances     = {depart: 0}
    predecesseurs = {depart: None}
    traites       = []

    # Les noeuds explorés
    explores      = 0

    # Le coût
    for sommet in grille.graphe_liste_adjacence:
        distances[sommet] = float('inf')

    distances_depart = {depart: 0}
    file = [(0, depart)]




    while file:
        distance_actuelle, sommet_actuel = heapq.heappop(file)

        if sommet_actuel in traites:
          continue

        traites.append(sommet)
        explores += 1

        if sommet_actuel == arrivee:
            break

        voisins= grille.voisins(sommet_actuel)

        for voisin, poids in voisins(sommet, grille):
            nouvelle_grille = distance actuelle + poids
            if nouveau_grille < distances[voisin]:
                distances[voisin]     = nouveau_grille
                predecesseurs[voisin] = sommet_actuel
                heapq.heappush(file, (nouveau_grille, voisin))
    # Le temps
    fin = time.perf_counter()
    chemin = remonte_chemin(depart, arrivee, predecesseurs)

    return {
        'chemin': chemin,
        'distance': distances[arrivee],
        'temps': fin - debut,
        'explores': explores
    }